In [ ]:
import wandb
api = wandb.Api()
run = api.run("/ahmedsamiratarjama-tarjama/chatterbox_new/runs/4i4pbul8")
print(run.history())

In [1]:
import torch

In [13]:
logits = torch.randn(20, 32)
# Sample soft categorical using reparametrization trick:
m = torch.nn.functional.gumbel_softmax(logits, tau=1, hard=True)
b = torch.nn.functional.softmax(logits)

/tmp/ipykernel_4158349/3678079169.py:4: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  b = torch.nn.functional.softmax(logits)


In [14]:
logits[0]

tensor([ 0.3092,  2.2015, -0.6340,  0.5206,  0.0975, -1.6806, -0.8524,  1.1634,
        -0.4147,  0.5203,  0.6686, -2.9958, -0.6665,  0.0529, -0.6461, -1.2679,
         0.0705, -0.9710,  1.1641,  2.5244, -2.1468, -0.5763, -1.0663,  1.4734,
         0.6927,  0.2429, -0.4072,  0.7048,  0.8309, -1.1984,  1.0752, -0.3059])

In [15]:
m[0]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [18]:
b[0]

tensor([0.0231, 0.1532, 0.0090, 0.0285, 0.0187, 0.0032, 0.0072, 0.0542, 0.0112,
        0.0285, 0.0331, 0.0008, 0.0087, 0.0179, 0.0089, 0.0048, 0.0182, 0.0064,
        0.0543, 0.2116, 0.0020, 0.0095, 0.0058, 0.0740, 0.0339, 0.0216, 0.0113,
        0.0343, 0.0389, 0.0051, 0.0497, 0.0125])

In [7]:
import json
import re
import uuid

def convert_to_transcribe_format(input_json):

    """
    Convert simple ASR JSON format to Amazon Transcribe-like format.
    """
    input_json = json.load(input_json) 
    job_id = str(uuid.uuid4())
    job_name = f"asr_job_{job_id}"
    account_id="1"
    word_duration=0.5
    transcription = input_json["transcription"].strip()

    # Split words and keep punctuation separate
    tokens = re.findall(r'\w+|[^\w\s]', transcription, re.UNICODE)

    items = []
    current_time = 0.0
    item_ids = []

    for idx, token in enumerate(tokens):
        item_ids.append(idx)

        if re.match(r'\w+', token):  # Word (pronunciation)
            start_time = current_time
            end_time = current_time + word_duration

            item = {
                "id": idx,
                "start_time": f"{start_time:.2f}",
                "end_time": f"{end_time:.2f}",
                "alternatives": [
                    {
                        "confidence": "1.0",
                        "content": token
                    }
                ],
                "type": "pronunciation"
            }

            current_time = end_time

        else:  # Punctuation
            item = {
                "id": idx,
                "alternatives": [
                    {
                        "confidence": "0.0",
                        "content": token
                    }
                ],
                "type": "punctuation"
            }

        items.append(item)

    # Full segment timing
    segment = {
        "id": 0,
        "transcript": transcription,
        "start_time": f"{0.00:.2f}",
        "end_time": f"{current_time:.2f}",
        "items": item_ids
    }

    output = {
        "jobName": job_name,
        "accountId": account_id,
        "results": {
            "transcripts": [
                {
                    "transcript": transcription
                }
            ],
            "items": items,
            "audio_segments": [segment]
        },
        "status": "COMPLETED"
    }

    return output


# ---------------- Example Usage ----------------

input_data = '''{
  "input_json": "{\"success\":true,\"filename\":\"audio (16).wav\",\"language\":\"auto-detected\",\"transcription\":\" معاك المحقق سعيد الشامسي انت متهم اليوم بالاعتذاء على المدعو جورج وضربه ما قوالك؟\",\"mode\":\"batch_processing\"}"
}'''
converted = convert_to_transcribe_format(input_data)

print(json.dumps(converted, ensure_ascii=False, indent=4))

AttributeError: 'str' object has no attribute 'read'

In [ ]:
import json
import re
import uuid

def convert_to_transcribe_format(input_data):

    outer_dict = json.loads(input_data)


    input_data = outer_dict
    
    job_id = str(uuid.uuid4())
    job_name = f"asr_job_{job_id}"
    account_id = "1"
    word_duration = 0.5
    print(input_data)

    transcription = input_data["transcription"].strip()

    tokens = re.findall(r'\w+|[^\w\s]', transcription, re.UNICODE)

    items = []
    current_time = 0.0
    item_ids = []

    for idx, token in enumerate(tokens):
        item_ids.append(idx)

        if re.match(r'\w+', token):
            start_time = current_time
            end_time = current_time + word_duration

            item = {
                "id": idx,
                "start_time": f"{start_time:.2f}",
                "end_time": f"{end_time:.2f}",
                "alternatives": [
                    {
                        "confidence": "1.0",
                        "content": token
                    }
                ],
                "type": "pronunciation"
            }

            current_time = end_time

        else:
            item = {
                "id": idx,
                "alternatives": [
                    {
                        "confidence": "0.0",
                        "content": token
                    }
                ],
                "type": "punctuation"
            }

        items.append(item)

    segment = {
        "id": 0,
        "transcript": transcription,
        "start_time": "0.00",
        "end_time": f"{current_time:.2f}",
        "items": item_ids
    }

    output = {
        "jobName": job_name,
        "accountId": account_id,
        "results": {
            "transcripts": [
                {
                    "transcript": transcription
                }
            ],
            "items": items,
            "audio_segments": [segment]
        },
        "status": "COMPLETED"
    }

    return output

In [35]:

input_data = '''
{
  "input_data": "{\"success\":true,\"filename\":\"audio (16).wav\",\"language\":\"auto-detected\",\"transcription\":\" معاك المحقق سعيد الشامسي انت متهم اليوم بالاعتذاء على المدعو جورج وضربه ما قوالك؟\",\"mode\":\"batch_processing\"}"
}
'''
converted = convert_to_transcribe_format(input_data)

print(json.dumps(converted, ensure_ascii=False, indent=4))

JSONDecodeError: Expecting ',' delimiter: line 3 column 20 (char 22)

In [33]:
import json

# Your raw input
input_data_str = '''
{
  "input_data": "{\\"success\\":true,\\"filename\\":\\"audio (16).wav\\",\\"language\\":\\"auto-detected\\",\\"transcription\\":\\" معاك المحقق سعيد الشامسي انت متهم اليوم بالاعتذاء على المدعو جورج وضربه ما قوالك؟\\",\\"mode\\":\\"batch_processing\\"}"
}
'''

# 1. Parse the outer string into a dictionary
outer_dict = json.loads(input_data_str)

# 2. Parse the inner string (which is inside the 'input_data' key)
inner_data = json.loads(outer_dict['input_data'])

# 3. Print the result as a clean, formatted JSON
print(json.dumps(inner_data, indent=2, ensure_ascii=False))

{
  "success": true,
  "filename": "audio (16).wav",
  "language": "auto-detected",
  "transcription": " معاك المحقق سعيد الشامسي انت متهم اليوم بالاعتذاء على المدعو جورج وضربه ما قوالك؟",
  "mode": "batch_processing"
}


In [ ]:
##
import json
import re
import uuid

def main(input_data: str):
    outer_dict = json.loads(input_data)
    inner_dict = json.loads(outer_dict["input_data"])
    return inner_dict